# Teste manual dos providers

Chama cada fonte de dados em [`src/providers/`](../src/providers/) para validar credenciais, rede e paths locais.

**Pré-requisitos**

1. Na raiz do projeto: `pip install -e ".[dev]"`
2. Copiar [`.env.example`](../.env.example) → `.env` e preencher (mínimo: `ANBIMA_*` para API ANBIMA; `UPTODATA_*` para ajustes BMF)
3. Kernel apontando para o mesmo venv

Abra este notebook na pasta `notebooks/` **ou** na raiz do repositório.

In [ ]:
import sys
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Execute na raiz do repo ou em notebooks/")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[misc, assignment]

from config import get_settings

get_settings.cache_clear()
settings = get_settings()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", settings.data_root)
print("LOG_LEVEL:", settings.log_level)
print("ANBIMA client_id definido:", bool(settings.anbima.client_id))
print("UpToData pasta IR:", settings.uptodata.pasta_interest_rate_base or "(vazio)")

In [ ]:
# Ajuste as datas de teste aqui
REF_DATE = (date.today() - timedelta(days=5)).isoformat()  # dia útil recente
DATES = [REF_DATE]
MES, ANO = int(REF_DATE[5:7]), int(REF_DATE[:4])

print("REF_DATE:", REF_DATE)
print("DATES:", DATES)


def preview_df(df: pd.DataFrame, title: str, n: int = 5) -> None:
    print("\n" + "=" * 60)
    print(f"{title} | shape={df.shape}")
    print("=" * 60)
    if df.empty:
        print("(DataFrame vazio)")
        return
    display(df.head(n))


def preview_list(items: list, title: str, n: int = 10) -> None:
    print("\n" + "=" * 60)
    print(f"{title} | len={len(items)}")
    print("=" * 60)
    print(items[:n])

## 1. ANBIMA API (OAuth)

Requer `ANBIMA_CLIENT_ID` e `ANBIMA_CLIENT_SECRET` no `.env`.

In [ ]:
from providers.anbima import AnbimaClient, MERCADO_SECUNDARIO_TPF, PROJECOES

client = AnbimaClient()

payload_ms = client.fetch_by_date(MERCADO_SECUNDARIO_TPF, REF_DATE)
print("mercado_secundario:", "OK" if payload_ms is not None else "sem dado (404 ou vazio)")
if payload_ms is not None:
    if isinstance(payload_ms, list):
        print("registros:", len(payload_ms))
        display(payload_ms[:2])
    else:
        display(payload_ms)

payload_proj = client.fetch_projecoes(MES, ANO)
print("\nprojecoes:", "OK" if payload_proj is not None else "sem dado")
if payload_proj is not None:
    display(payload_proj if not isinstance(payload_proj, list) else payload_proj[:3])

## 2. Feriados (XLS público ANBIMA)

Sem credenciais. URL em `FERIADOS_XLS_URL`.

In [ ]:
from providers.feriados import fetch_feriados

feriados = fetch_feriados()
preview_list(feriados, "Feriados nacionais (ISO)")

## 3. BCB — liquidações (ZIP NegE)

Download público; pode demorar conforme a rede.

In [ ]:
from providers.bcb import build_negociacoes_url, fetch_negociacoes_bruto_por_datas

print("URL:", build_negociacoes_url(REF_DATE))

df_bcb = fetch_negociacoes_bruto_por_datas(DATES, date_column="DATA MOV")
preview_df(df_bcb, "BCB NegE (bruto)")

## 4. Tesouro Nacional — resultados de leilões

API pública; busca por ano derivado de `DATES`.

In [ ]:
from providers.tesouro import get_resultados, get_resultados_by_dates

resp_ano = get_resultados(ano=int(REF_DATE[:4]))
n_reg = len(resp_ano.get("registros", [])) if isinstance(resp_ano, dict) else 0
print(f"get_resultados(ano={REF_DATE[:4]}): {n_reg} registros")

rows = get_resultados_by_dates(DATES)
preview_list(rows, "get_resultados_by_dates (amostra dict)")
if rows:
    display(pd.DataFrame(rows).head(5))

## 5. SIDRA / IBGE — IPCA (sidrapy)

Snapshot; não usa lista de datas.

In [ ]:
from providers.sidra import SidraIpcaClient

df_sidra = SidraIpcaClient().fetch_table_ipca()
preview_df(df_sidra, "SIDRA IPCA (bruto sidrapy)")

## 6. UpToData — ajustes BMF (arquivos locais)

Requer no `.env`:

- `UPTODATA_PASTA_INTEREST_RATE_BASE`
- `UPTODATA_ARQUIVO_INTEREST_RATE_BASE`

Sem paths válidos retorna DataFrame vazio.

In [ ]:
from providers.uptodata import scrap_ajustes_bmf, scrap_ajustes_bmf_for_dates

cfg = settings.uptodata
print("pasta:", cfg.pasta_interest_rate_base or "(não configurado)")
print("prefixo arquivo:", cfg.arquivo_interest_rate_base or "(não configurado)")

df_upto_one = scrap_ajustes_bmf(REF_DATE)
preview_df(df_upto_one, f"UpToData um dia ({REF_DATE})")

df_upto = scrap_ajustes_bmf_for_dates(DATES)
preview_df(df_upto, "UpToData for_dates")

## Resumo dos contratos (referência)

| Provider | Contrato em `contracts` |
|----------|-------------------------|
| BCB, UpToData | `DateRangeDataFrameFetcher` |
| Feriados | `SnapshotDateListFetcher` |
| Tesouro | `DateRangeRecordFetcher` |
| SIDRA | `SidraIpcaProvider` |
| ANBIMA | `AnbimaFeedClient` |